# Event Analysis Queries
* Ryan Kazmerik
* April 29, 2025

Proposed queries for basic event analysis, such as the number of events of a user, user event flow in a time window and additional queries that could be interesting to analyze the user behaviour.

In [120]:
import awswrangler as wr
import pandas as pd

In [121]:
DATABASE = "events_db"
S3_BUCKET = "s3://athena-query-results-806a5225/results/"

## Basic Queries
### Events by User
Let's see who the top 10 most active users are in the past 3 months by querying to see the number of events by user.

We also enriched each record with a `session_id` which stays the same for a 1-hour window in order to track events within a single session. 

In [152]:
df_top_users = wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        SELECT user_id, COUNT(*) AS total_events, COUNT(DISTINCT session_id) AS total_sessions
        FROM events_db.events
        WHERE event_date >= CAST(current_date - INTERVAL '90' DAY AS VARCHAR)
        GROUP BY user_id
        ORDER BY total_events DESC;
    """
)

df_top_users.head(10)

,user_id,total_events,total_sessions
0,198531,283,268
1,743194,274,245
2,347469,274,265
3,583708,271,253
4,634782,270,252
5,347267,269,248
6,598330,268,253
7,560664,267,249
8,110760,267,251
9,180707,266,244


> #### **Insight:** The number of events compared to sessions is relatively close for each user, suggesting that users visit our app often, but only make a few clicks before leaving.

### User Event Flow in a Time Window
Let's dig into the top user and see what activities they've been up to in the past 7 days

In [132]:
top_user_id = df_top_users.iloc[0].user_id

df_top_user_history = wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= f"""
        SELECT event_date, event_timestamp, page, event_type
        FROM events_db.events
        WHERE event_date >= CAST(current_date - INTERVAL '7' DAY AS VARCHAR)
        AND user_id = '{top_user_id}'
        ORDER BY event_timestamp DESC;
    """
)

df_top_user_history.head(100)


,event_date,event_timestamp,page,event_type
0,2025-05-01,2025-05-01 04:21:07,sizzle,click
1,2025-05-01,2025-05-01 00:31:16,demo,search
2,2025-04-30,2025-04-30 08:00:11,pricing,view
3,2025-04-30,2025-04-30 02:37:51,demo,view
4,2025-04-29,2025-04-29 21:09:09,promotions,error
5,2025-04-29,2025-04-29 17:19:51,pricing,view
6,2025-04-29,2025-04-29 09:08:54,promotions,view
7,2025-04-28,2025-04-28 22:41:37,tutorials,view
8,2025-04-28,2025-04-28 14:48:16,purchase,error
9,2025-04-28,2025-04-28 09:25:10,purchase,error


> #### **Insight:** This user has been checking out our articles, demo & tutorial pages quite often, suggesting they are interested in learning about the product, but they haven't made an actual purchase - they might be a good candidate for promotion marketing to give them a nudge to make an actual purchase.

## Additional Queries

### What Content is Driving Purchases?
* Objective: New Customer Acquisition
* Target Audience: Revenue Team 

One thing we're curious about is what's driving traffic to our purchase page? One way to investigate this may be to see what pages users were viewing before they decided to visit the purchase page to help understand what content is making users want to purchase.


In [124]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH user_journeys AS (
            SELECT user_id, event_type, event_timestamp, page,
            LAG(page) OVER (PARTITION BY user_id ORDER BY event_timestamp) AS previous_page
            FROM events_db.events
        ),
        product_visits AS (
            SELECT previous_page, COUNT(*) AS product_purchases
            FROM user_journeys
            WHERE page = 'purchase'
            AND previous_page != 'purchase'
            AND event_type = 'submit'
            GROUP BY previous_page
        )
        SELECT previous_page, product_purchases
        FROM product_visits
        ORDER BY product_purchases DESC;
    """
).head(10)

,previous_page,product_purchases
0,articles,314
1,promotions,312
2,login,306
3,pricing,300
4,demo,299
5,tutorials,286
6,sizzle,268


> #### **Insight:** The articles we've been publishing about our product are resulting in the most purchases, so while they take time to produce they are highly effective. Moreover, the sizzle video that we produced has not been very effective, and cost a lot more to produce than writing the articles.

What also may be interesting is what content is NOT driving purchases, or what content are customers looking at right before they drop off and exit the app without making a purchase.

To do this, we can use our `session_id` variable we generated during event enrichment to see the **last page visited in each session** to understand where users most often drop off.

In [149]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH ranked_events AS (
            SELECT user_id, session_id, page, event_timestamp, ROW_NUMBER() OVER (PARTITION BY session_id
            ORDER BY event_timestamp DESC
            ) AS rank
            FROM events_db.events
            WHERE event_date >= CAST(current_date - INTERVAL '30' DAY AS VARCHAR)
        ),
            last_events AS (
                SELECT *
                FROM ranked_events
                WHERE rank = 1
            )
            SELECT page AS drop_off_page, COUNT(*) AS drop_off_count
            FROM last_events
            GROUP BY page
            ORDER BY drop_off_count DESC;
    """
).head(20)

,drop_off_page,drop_off_count
0,login,992
1,pricing,991
2,tutorials,989
3,articles,989
4,purchase,943
5,sizzle,940
6,promotions,931
7,demo,928


> #### **Insight:** The login page, and the pricing page are the two areas where most users drop off. Perhaps digging into if users are having trouble logging into your app (we'll do that later) would be wise. And perhaps the pricing page needs a revamp as it seems it's scaring customers away. 

### Which Users are Losing Interest?

* Objective : Customer Retention
* Target Audience : Customer Success Team

It may be helpful to identify users who used to use our app a lot, but their activity is dropping off. We could reach out to them to prevent them from churning from our service all together.

In [169]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH past_active_users AS (
            SELECT user_id, COUNT(*) as events_in_past_1m
            FROM events_db.events
            WHERE event_date >= CAST(current_date - INTERVAL '30' DAY AS VARCHAR)
            GROUP BY user_id
        ),
        recent_inactive_users AS (
            SELECT user_id, COUNT(*) as events_today
            FROM events_db.events
            WHERE event_date >= CAST(current_date - INTERVAL '1' DAY AS VARCHAR)
            GROUP BY user_id
        )
        SELECT p.user_id, events_in_past_1m, events_today
        FROM past_active_users p
        LEFT JOIN recent_inactive_users r ON p.user_id = r.user_id
        WHERE r.user_id IS NULL
        ORDER BY events_in_past_1m DESC;
    """
).fillna(0).head(10)

,user_id,events_in_past_1m,events_today
0,360763,92,0
1,485924,90,0
2,693778,83,0
3,745834,83,0
4,767489,79,0
5,599617,79,0
6,589999,76,0
7,691110,75,0
8,170276,73,0
9,592999,69,0


> #### **Insight:** These users have demonstrated high activity in the past but have not accessed the app recently. Perhaps they are not interested in the app anymore, or are having trouble accessing the app? It may be wise to monitor these customers going forward to see if they come back and reach out to them if they don't.

### What Platform Drives the most Engagement?

* Objective : Channel Prioritization
* Target Audience : Product Team

We can look at which platform helps drive the most engagement with it's users by measuring the amount of events by device type and browser type

In [177]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH recent_events AS (
            SELECT browser, device, date_trunc('month', event_timestamp) AS event_month
            FROM events_db.events
            WHERE event_date >= CAST(current_date - INTERVAL '180' DAY AS VARCHAR)
        ),
        monthly_event_counts AS (
            SELECT browser, device, event_month, COUNT(*) AS monthly_events
            FROM recent_events
            GROUP BY device, browser, event_month
        )
        SELECT browser, device, ROUND(AVG(monthly_events), 2) AS avg_events_per_month
        FROM monthly_event_counts
        GROUP BY device, browser
        ORDER BY avg_events_per_month DESC;
    """
).head(10)

,browser,device,avg_events_per_month
0,Safari,Mobile,599.71
1,Edge,Mobile,598.00
2,Edge,Desktop,592.29
3,Firefox,Desktop,592.14
4,Chrome,Desktop,592.00
5,Safari,Desktop,587.57
6,Safari,Tablet,586.71
7,Firefox,Tablet,583.71
8,Chrome,Tablet,583.29
9,Chrome,Mobile,579.86


> #### **Insight:** While the difference is slight, this would suggest that Mobile users (more specifically iPhone users) are our most active user, and that fewer of our customers are accessing our app via tablet. This data could help the Product team prioritize features for development in their backlog for each channel.

### Who's Having Trouble Logging In?

* Objective : Reduce Friction
* Target Audience : Support Team

Let's see if any users are consistently having trouble logging in in the past month.

In [178]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH failed_logins AS (
            SELECT user_id, device, browser, event_date
            FROM events_db.events
            WHERE page = 'login' AND event_type = 'error'
            AND date_parse(event_date, '%Y-%m-%d') >= current_date - interval '1' month
        )
        SELECT user_id, COUNT(*) AS failed_login_attempts
        FROM failed_logins
        GROUP BY user_id
        ORDER BY failed_login_attempts DESC;
    """
).head(10)

,user_id,failed_login_attempts
0,187161,6
1,180707,6
2,195702,6
3,662162,5
4,789230,5
5,583708,5
6,749178,5
7,713613,5
8,251437,5
9,417736,5


> #### **Insight:** This would suggest that we do have users with trouble logging in. Perhaps we can proactively reach out to them and help them access our app, before they get frustrated and lose interest all together.

### These queries are simple but actionable and help us determine our next best action for improving our app, user experience and purchase conversion rates.